# Placement Pulse — Placement Analytics & Prediction
**Complete Google Colab notebook**

This notebook performs data loading, cleaning, preprocessing, exploratory analysis, statistical analysis, model training, evaluation, visualizations, interpretation, and model export.

**Important:** The supplied 5,000-row dataset is synthetic. Results describe this dataset only and should not be presented as actual college placement outcomes.

**Target:** `Placement_Status`  
**Leakage prevention:** `Student_ID`, `Offer_Count`, and `Salary` are excluded from the placement classifier because they identify students or are outcomes/known only after placement.


## 1. Install and import libraries

In [ ]:
# Colab usually has these libraries installed. Uncomment if needed.
# %pip install -q pandas numpy matplotlib seaborn scikit-learn scipy joblib

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency, mannwhitneyu, spearmanr, pearsonr
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, learning_curve
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    balanced_accuracy_score, matthews_corrcoef, cohen_kappa_score,
    log_loss, brier_score_loss
)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
import joblib

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

print("Libraries imported successfully.")

## 2. Upload the dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Please upload a CSV dataset.")
DATA_PATH = csv_files[0]
print("Using:", DATA_PATH)

df_raw = pd.read_csv(DATA_PATH)
print("Raw dataset shape:", df_raw.shape)
display(df_raw.head())

## 3. Initial data inspection

In [ ]:
print("Shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())
print("\nData types:")
display(df_raw.dtypes.to_frame("dtype"))
print("\nDataset information:")
df_raw.info()
print("\nDescriptive statistics:")
display(df_raw.describe(include="all").T)
print("\nMissing values:")
display(df_raw.isna().sum().to_frame("missing_count").assign(
    missing_percent=lambda x: (x["missing_count"] / len(df_raw) * 100).round(2)
))
print("\nDuplicate rows:", df_raw.duplicated().sum())

## 4. Data cleaning and validation

In [ ]:
df = df_raw.copy()

# Normalize column names and text values.
df.columns = df.columns.str.strip()
required = [
    "Student_ID", "Academic_Performance", "Coding_Score", "Project_Score",
    "Certifications", "Internships", "Communication_Score",
    "Placement_Status", "Offer_Count", "Salary", "Department", "Graduation_Year"
]
missing_cols = [c for c in required if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing expected columns: {missing_cols}")

# Remove fully duplicated records.
before = len(df)
df = df.drop_duplicates().copy()
print("Removed duplicate rows:", before - len(df))

# Convert numeric columns safely; invalid values become NaN.
numeric_cols = [
    "Academic_Performance", "Coding_Score", "Project_Score",
    "Certifications", "Internships", "Offer_Count", "Salary", "Graduation_Year"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Standardize target labels.
df["Placement_Status"] = (
    df["Placement_Status"].astype("string").str.strip().str.lower()
    .replace({
        "placed": "Placed", "yes": "Placed", "1": "Placed", "true": "Placed",
        "not placed": "Not Placed", "not_placed": "Not Placed",
        "unplaced": "Not Placed", "no": "Not Placed", "0": "Not Placed", "false": "Not Placed"
    })
)

# Normalize department strings.
df["Department"] = df["Department"].astype("string").str.strip().str.upper()

# Remove rows without a valid target; never impute the outcome.
valid_target = df["Placement_Status"].isin(["Placed", "Not Placed"])
print("Rows with invalid/missing target:", int((~valid_target).sum()))
df = df.loc[valid_target].copy()

# Ensure numeric ranges are plausible. Invalid values are converted to missing,
# then handled by the preprocessing pipeline.
range_limits = {
    "Academic_Performance": (0, 100),
    "Coding_Score": (0, 100),
    "Project_Score": (0, 100),
    "Communication_Score": (0, 100),
    "Certifications": (0, None),
    "Internships": (0, None),
    "Offer_Count": (0, None),
    "Salary": (0, None),
    "Graduation_Year": (1900, 2100)
}
for col, (low, high) in range_limits.items():
    df.loc[df[col] < low, col] = np.nan
    if high is not None:
        df.loc[df[col] > high, col] = np.nan

# Salary is meaningful only for placed students; missing salary for unplaced
# students is expected and is not replaced with zero.
print("Cleaned shape:", df.shape)
display(df.head())
print("\nMissing values after cleaning:")
display(df.isna().sum().to_frame("missing_count"))

## 5. Exploratory Data Analysis (EDA)

In [ ]:
# 5.1 Target distribution
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="Placement_Status", order=["Not Placed", "Placed"])
ax.set_title("Placement Status Distribution")
ax.set_xlabel("Placement status")
ax.set_ylabel("Number of students")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

placement_counts = df["Placement_Status"].value_counts()
placement_rates = (placement_counts / len(df) * 100).round(2)
display(pd.DataFrame({"count": placement_counts, "percent": placement_rates}))

# 5.2 Numeric feature distributions
eda_numeric = [
    "Academic_Performance", "Coding_Score", "Project_Score",
    "Certifications", "Internships", "Communication_Score"
]
df[eda_numeric].hist(figsize=(16, 10), bins=25, edgecolor="black")
plt.suptitle("Distributions of Student Attributes", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

# 5.3 Boxplots by placement outcome
for col in eda_numeric:
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="Placement_Status", y=col,
                order=["Not Placed", "Placed"])
    plt.title(f"{col} by Placement Status")
    plt.tight_layout()
    plt.show()

# 5.4 Placement rate by department
dept_rate = (df.groupby("Department", dropna=False)["Placement_Status"]
             .apply(lambda s: (s == "Placed").mean() * 100)
             .sort_values(ascending=False))
plt.figure(figsize=(10, 5))
sns.barplot(x=dept_rate.index.astype(str), y=dept_rate.values)
plt.title("Placement Rate by Department")
plt.xlabel("Department")
plt.ylabel("Placement rate (%)")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()
display(dept_rate.rename("placement_rate_percent").to_frame().round(2))

# 5.5 Placement rate by graduation year
year_rate = (df.groupby("Graduation_Year")["Placement_Status"]
             .apply(lambda s: (s == "Placed").mean() * 100).dropna())
plt.figure(figsize=(9, 5))
sns.lineplot(x=year_rate.index, y=year_rate.values, marker="o")
plt.title("Placement Rate by Graduation Year")
plt.xlabel("Graduation year")
plt.ylabel("Placement rate (%)")
plt.tight_layout()
plt.show()

# 5.6 Correlation heatmap (numeric fields only)
corr_cols = eda_numeric + ["Offer_Count", "Salary", "Graduation_Year"]
plt.figure(figsize=(12, 8))
sns.heatmap(df[corr_cols].corr(numeric_only=True), annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=.5)
plt.title("Correlation Heatmap — Numeric Variables")
plt.tight_layout()
plt.show()

## 6. Salary and offers analysis

In [ ]:
# Analyze salary only for placed students with a recorded positive salary.
placed = df[(df["Placement_Status"] == "Placed") & df["Salary"].notna()].copy()

print("Placed students:", int((df["Placement_Status"] == "Placed").sum()))
print("Placed students with salary recorded:", len(placed))
if len(placed):
    display(placed["Salary"].describe().to_frame("Salary"))
    plt.figure(figsize=(9, 5))
    sns.histplot(placed["Salary"], bins=25, kde=True)
    plt.title("Salary Distribution Among Placed Students")
    plt.xlabel("Salary (dataset units)")
    plt.tight_layout()
    plt.show()

    for col in ["Coding_Score", "Academic_Performance", "Project_Score",
                "Communication_Score", "Certifications", "Internships"]:
        temp = placed[[col, "Salary"]].dropna()
        if len(temp) >= 3:
            plt.figure(figsize=(8, 5))
            sns.regplot(data=temp, x=col, y="Salary",
                        scatter_kws={"alpha": .45}, line_kws={"linewidth": 2})
            plt.title(f"{col} vs Salary (Placed Students)")
            plt.tight_layout()
            plt.show()
            rho, p = spearmanr(temp[col], temp["Salary"])
            print(f"Spearman correlation {col} vs Salary: rho={rho:.3f}, p={p:.4g}")

    offer_summary = placed.groupby("Offer_Count", dropna=False)["Salary"].agg(
        ["count", "mean", "median"]
    )
    display(offer_summary.round(3))
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=placed, x="Offer_Count", y="Salary")
    plt.title("Salary Distribution by Number of Offers")
    plt.xlabel("Number of offers")
    plt.tight_layout()
    plt.show()
else:
    print("No usable salary records found; salary plots are skipped.")

## 7. Statistical comparisons

In [ ]:
# Descriptive comparisons are associations, not proof of causation.
print("Mean attribute values by placement status:")
display(df.groupby("Placement_Status")[eda_numeric].mean().T.round(2))

# Mann-Whitney U tests compare distributions for each numeric feature.
# These tests do not establish that a feature causes placement.
test_rows = []
for col in eda_numeric:
    a = df.loc[df["Placement_Status"] == "Placed", col].dropna()
    b = df.loc[df["Placement_Status"] == "Not Placed", col].dropna()
    if len(a) and len(b):
        stat, p = mannwhitneyu(a, b, alternative="two-sided")
        test_rows.append({"feature": col, "placed_median": a.median(),
                          "not_placed_median": b.median(), "U_statistic": stat, "p_value": p})
display(pd.DataFrame(test_rows).sort_values("p_value").round(4))

# Department vs placement chi-square test
contingency = pd.crosstab(df["Department"], df["Placement_Status"])
display(contingency)
if contingency.shape[0] > 1 and contingency.shape[1] > 1:
    chi2, p, dof, expected = chi2_contingency(contingency)
    print(f"Department vs placement: chi2={chi2:.3f}, dof={dof}, p={p:.4g}")
else:
    print("Not enough categories for chi-square test.")

## 8. Prepare features and prevent data leakage

In [ ]:
TARGET = "Placement_Status"

# Exclude identifiers and post-outcome fields from the placement model.
LEAKAGE_OR_ID_COLS = ["Student_ID", "Offer_Count", "Salary"]
FEATURE_COLS = [
    "Academic_Performance", "Coding_Score", "Project_Score",
    "Certifications", "Internships", "Communication_Score",
    "Department", "Graduation_Year"
]

X = df[FEATURE_COLS].copy()
y = df[TARGET].map({"Not Placed": 0, "Placed": 1})

# Drop rows with any target mapping failure (normally none after validation).
valid = y.notna()
X, y = X.loc[valid], y.loc[valid].astype(int)

numeric_features = [
    "Academic_Performance", "Coding_Score", "Project_Score",
    "Certifications", "Internships", "Graduation_Year"
]
categorical_features = ["Department"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    # Scaling is not required for Random Forest, but retained for consistency
    # and potential comparison with scale-sensitive models.
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

## 9. Train the Random Forest model

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf)
])

model.fit(X_train, y_train)
print("Random Forest training completed.")

## 10. Evaluate the model — accuracy and other metrics

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, zero_division=0),
    "Recall": recall_score(y_test, y_pred, zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_proba) if y_test.nunique() == 2 else np.nan,
    "PR-AUC (Average Precision)": average_precision_score(y_test, y_proba)
        if y_test.nunique() == 2 else np.nan,
    "Matthews Correlation": matthews_corrcoef(y_test, y_pred),
    "Cohen Kappa": cohen_kappa_score(y_test, y_pred),
    "Log Loss": log_loss(y_test, model.predict_proba(X_test), labels=[0, 1])
}
metrics_df = pd.DataFrame.from_dict(metrics, orient="index", columns=["score"])
display(metrics_df.round(4))

print("\nClassification report:")
print(classification_report(
    y_test, y_pred, labels=[0, 1],
    target_names=["Not Placed", "Placed"], zero_division=0
))

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                              display_labels=["Not Placed", "Placed"])
disp.plot(values_format="d")
plt.title("Confusion Matrix — Random Forest")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp) if (tn + fp) else np.nan
sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
print(f"Specificity (Not Placed recall): {specificity:.4f}")
print(f"Sensitivity (Placed recall): {sensitivity:.4f}")

## 11. ROC curve and Precision–Recall curve

In [ ]:
if y_test.nunique() == 2:
    fpr, tpr, thresholds = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, label=f"Random Forest (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Random baseline")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

    precision, recall, pr_thresholds = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    baseline = y_test.mean()

    plt.figure(figsize=(7, 6))
    plt.plot(recall, precision, label=f"Random Forest (AP = {ap:.3f})")
    plt.axhline(baseline, linestyle="--", label=f"Positive prevalence = {baseline:.3f}")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision–Recall Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("ROC and PR curves require both classes in the test split.")

## 12. Prediction probability distribution and calibration

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(x=y_proba, hue=y_test.map({0: "Not Placed", 1: "Placed"}),
             bins=20, stat="density", common_norm=False, element="step")
plt.title("Predicted Placement Probability by Actual Outcome")
plt.xlabel("Predicted probability of placement")
plt.tight_layout()
plt.show()

if y_test.nunique() == 2:
    prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10, strategy="quantile")
    plt.figure(figsize=(7, 6))
    plt.plot(prob_pred, prob_true, marker="o", label="Random Forest")
    plt.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed placement fraction")
    plt.title("Calibration Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Brier score:", round(brier_score_loss(y_test, y_proba), 4))

## 13. Cross-validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc",
    "average_precision": "average_precision"
}
cv_results = cross_validate(
    model, X_train, y_train, cv=cv, scoring=scoring,
    n_jobs=-1, return_train_score=False
)
cv_summary = pd.DataFrame({
    metric: [cv_results[f"test_{metric}"].mean(), cv_results[f"test_{metric}"].std()]
    for metric in scoring
}, index=["mean", "std"]).T
display(cv_summary.round(4))

plt.figure(figsize=(10, 5))
cv_long = pd.DataFrame({
    "fold": np.tile(np.arange(1, 6), len(scoring)),
    "metric": np.repeat(list(scoring.keys()), 5),
    "score": np.concatenate([cv_results[f"test_{m}"] for m in scoring])
})
sns.boxplot(data=cv_long, x="metric", y="score")
plt.title("5-Fold Cross-Validation Score Distribution")
plt.xlabel("Metric")
plt.ylabel("Score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 14. Learning curve

In [ ]:
train_sizes, train_scores, valid_scores = learning_curve(
    model, X_train, y_train, cv=cv, scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1, shuffle=True, random_state=RANDOM_STATE
)
train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
valid_mean = valid_scores.mean(axis=1)
valid_std = valid_scores.std(axis=1)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_mean, marker="o", label="Training F1")
plt.fill_between(train_sizes, train_mean-train_std, train_mean+train_std, alpha=.15)
plt.plot(train_sizes, valid_mean, marker="o", label="Cross-validation F1")
plt.fill_between(train_sizes, valid_mean-valid_std, valid_mean+valid_std, alpha=.15)
plt.xlabel("Number of training examples")
plt.ylabel("F1 score")
plt.title("Learning Curve")
plt.legend()
plt.tight_layout()
plt.show()

## 15. Feature importance

In [ ]:
# Random Forest impurity importance after preprocessing.
fitted_preprocessor = model.named_steps["preprocessor"]
feature_names = fitted_preprocessor.get_feature_names_out()
importances = model.named_steps["classifier"].feature_importances_

importance_df = (pd.DataFrame({"feature": feature_names, "importance": importances})
                 .sort_values("importance", ascending=False))
display(importance_df.head(20).round(4))

plt.figure(figsize=(10, 7))
sns.barplot(data=importance_df.head(20), x="importance", y="feature")
plt.title("Top 20 Random Forest Feature Importances")
plt.xlabel("Mean decrease in impurity")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Permutation importance on held-out data, measured using F1.
perm = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE,
    scoring="f1", n_jobs=-1
)
perm_df = (pd.DataFrame({"feature": X_test.columns,
                         "importance_mean": perm.importances_mean,
                         "importance_std": perm.importances_std})
           .sort_values("importance_mean", ascending=False))
display(perm_df.round(4))

plt.figure(figsize=(9, 5))
sns.barplot(data=perm_df, x="importance_mean", y="feature")
plt.title("Permutation Importance (Test F1)")
plt.xlabel("Mean change in F1 after shuffling")
plt.ylabel("Original feature")
plt.tight_layout()
plt.show()

## 16. Skill gap analysis

In [ ]:
# Compare average skill scores between placed and not-placed students.
skill_cols = ["Academic_Performance", "Coding_Score", "Project_Score", "Communication_Score"]
skill_means = df.groupby("Placement_Status")[skill_cols].mean().T
display(skill_means.round(2))

gap = (skill_means["Placed"] - skill_means["Not Placed"]).sort_values()
plt.figure(figsize=(9, 5))
sns.barplot(x=gap.values, y=gap.index)
plt.axvline(0, linestyle="--")
plt.title("Difference in Mean Scores: Placed minus Not Placed")
plt.xlabel("Mean score difference (dataset points)")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Placement rate by score band; use descriptive bins.
for col in ["Coding_Score", "Academic_Performance", "Project_Score", "Communication_Score"]:
    tmp = df[[col, "Placement_Status"]].dropna().copy()
    if not tmp.empty:
        tmp["Score_Band"] = pd.cut(tmp[col], bins=[-0.001, 40, 60, 75, 90, 100],
                                   labels=["0–40", "41–60", "61–75", "76–90", "91–100"])
        band_rate = tmp.groupby("Score_Band", observed=False)["Placement_Status"].apply(
            lambda s: (s == "Placed").mean() * 100
        )
        plt.figure(figsize=(8, 4))
        sns.barplot(x=band_rate.index.astype(str), y=band_rate.values)
        plt.title(f"Placement Rate by {col} Band")
        plt.xlabel("Score band")
        plt.ylabel("Placement rate (%)")
        plt.tight_layout()
        plt.show()

## 17. Compare a baseline model

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent"))
])
baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

comparison = pd.DataFrame([
    {"Model": "Majority-class baseline",
     "Accuracy": accuracy_score(y_test, baseline_pred),
     "Balanced Accuracy": balanced_accuracy_score(y_test, baseline_pred),
     "Precision": precision_score(y_test, baseline_pred, zero_division=0),
     "Recall": recall_score(y_test, baseline_pred, zero_division=0),
     "F1": f1_score(y_test, baseline_pred, zero_division=0)},
    {"Model": "Random Forest",
     "Accuracy": accuracy_score(y_test, y_pred),
     "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
     "Precision": precision_score(y_test, y_pred, zero_division=0),
     "Recall": recall_score(y_test, y_pred, zero_division=0),
     "F1": f1_score(y_test, y_pred, zero_division=0)}
])
display(comparison.set_index("Model").round(4))

## 18. Try a new student prediction

In [ ]:
# Edit the values below to predict for a hypothetical student.
# Use the same feature names and approximate scales as the dataset.
new_student = pd.DataFrame([{
    "Academic_Performance": 78,
    "Coding_Score": 82,
    "Project_Score": 75,
    "Certifications": 3,
    "Internships": 1,
    "Communication_Score": 80,
    "Department": "CSE",
    "Graduation_Year": 2027
}])

pred_class = model.predict(new_student)[0]
pred_probability = model.predict_proba(new_student)[0, 1]
print("Predicted class:", "Placed" if pred_class == 1 else "Not Placed")
print(f"Predicted placement probability: {pred_probability:.2%}")
print("This is a model estimate, not a guarantee or a causal assessment.")
display(new_student)

## 19. Export the trained model and evaluation results

In [ ]:
MODEL_PATH = "placement_pulse_random_forest.joblib"
METRICS_PATH = "placement_pulse_metrics.csv"
IMPORTANCE_PATH = "placement_pulse_feature_importance.csv"

joblib.dump(model, MODEL_PATH)
metrics_df.to_csv(METRICS_PATH)
importance_df.to_csv(IMPORTANCE_PATH, index=False)

print("Saved files:")
print(MODEL_PATH, METRICS_PATH, IMPORTANCE_PATH)

from google.colab import files
files.download(MODEL_PATH)
files.download(METRICS_PATH)
files.download(IMPORTANCE_PATH)

## 20. Summary and interpretation checklist

Use the actual output from your notebook to write your report:

1. **Dataset profile:** number of rows, columns, missing values, duplicate rows and placement distribution.
2. **Placement patterns:** compare placement rates across departments and graduation years, noting small group sizes.
3. **Skill comparisons:** compare score distributions and medians for placed and not-placed students.
4. **Salary analysis:** restrict it to placed students with valid salary records; report the dataset's salary unit only if known.
5. **Model evaluation:** report accuracy alongside precision, recall, F1, balanced accuracy, ROC-AUC and PR-AUC. With class imbalance, accuracy alone can be misleading.
6. **Model validation:** compare the held-out test metrics with cross-validation results and inspect the learning curve.
7. **Feature interpretation:** treat feature importance as predictive association, not proof of cause.
8. **Action plan:** use observed gaps to propose coding practice, project mentoring, internships and communication practice, then evaluate those interventions prospectively.

### Important limitations
- This is a synthetic dataset, so its results do not establish real placement trends at a college.
- A random train/test split evaluates performance on similar data; it does not guarantee performance on future graduating batches.
- Avoid using `Offer_Count` and `Salary` as predictors of whether a student gets placed: they are downstream outcomes and would leak information.
- Do not interpret model probabilities as certainty. Consider fairness, calibration, external validation and human review before any real-world use.
